# Integrales-04 — Le rung elliptique du sunrise

Le carnet 02 intégrait le sunrise à masses égales en $d = 2$ par quatre voies
accordées — sans jamais nommer la structure cachée. Le carnet 03 faisait diverger
la bulle en $d = 4$. Celui-ci revient au sunrise et **exhibe sa courbe
elliptique** — non pas en la recopiant d'une référence, mais en la laissant
sortir du calcul : à un paramètre de Feynman fixé, l'intégrale interne se fait
en fonctions élémentaires, et ce qui reste vit sur la racine d'une **quartique**
$y^2 = b(\chi)\,c(\chi)$. Le classement BootLoops (polylogarithmes →
**elliptique** → K3 → Calabi-Yau) franchit ici son premier rang.

Le programme :

1. **rappel et ancre** — la forme paramétrique vérifiée du carnet 02,
   $I(s)=\iint 1/(U+s\,x_1x_2x_3)$, et l'ancre $I(0)=\frac{4}{\sqrt3}\mathrm{Cl}_2(\pi/3)$ ;
2. **la voie interne** — l'intégrale en $x$ faite à la main (un `artanh`),
   le reste donnant $I(s)$ comme intégrale **de $\log$ contre $d\chi/y$** :
   un dilogarithme elliptique en formation ;
3. **la courbe** — racines, dégénérescences ($s\to 0$ : une racine s'échappe
   à l'infini et la courbe devient le monde $\mathrm{Cl}_2$ du carnet 01 ;
   $s\to\infty$ : deux racines se collisionnent, pointe de la courbe) ;
4. **les périodes** — mesurées par quadrature, réduites à $K(k)$, module
   mesuré par birapport des racines ;
5. **$\tau(s)$ court** — la courbe dépend de l'échelle, contrairement au
   monde des constantes des carnets 01-02.

Comme toujours : outil réel (`System.Math`, double précision), chaque affirmation
quantitative vit dans une sortie commise, les refus sont documentés.

## 1. Rappel : la forme vérifiée du carnet 02

$$I(s) = \int_0^1\!\!\int_0^{1-x_1} \frac{dx_2\,dx_1}{U + s\,x_1x_2x_3},
\qquad U = x_1x_2+x_1x_3+x_2x_3, \quad x_3 = 1-x_1-x_2,$$

vérifiée au carnet 02 contre trois autres voies (rectangle $(u,v)$, Cheng–Wu,
hyperbolique) — avec l'ancre exacte $I(0) = \frac{4}{\sqrt3}\mathrm{Cl}_2(\pi/3)$,
le triangle du carnet 01 évalué à la cinématique $J(1,1,1)$.

In [1]:
using System;
using System.Globalization;
using System.Linq;
using System.Collections.Generic;
using System.Diagnostics;

CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;
CultureInfo.CurrentUICulture = CultureInfo.InvariantCulture;

// Quadrature de Gauss-Legendre : les nœuds sont strictement intérieurs.
(double[] x, double[] w) Gauss(int n)
{
    var x = new double[n];
    var w = new double[n];
    for (int i = 0; i < (n + 1) / 2; i++)
    {
        double z = Math.Cos(Math.PI * (i + 0.75) / (n + 0.5));
        double pp = 0;
        for (int iteration = 0; iteration < 30; iteration++)
        {
            double p = 1, previous = 0;
            for (int j = 1; j <= n; j++)
            {
                double next = ((2.0 * j - 1) * z * p - (j - 1) * previous) / j;
                previous = p;
                p = next;
            }
            pp = n * (z * p - previous) / (z * z - 1);
            double nextZ = z - p / pp;
            if (Math.Abs(nextZ - z) < 1e-15) { z = nextZ; break; }
            z = nextZ;
        }
        double weight = 2 / ((1 - z * z) * pp * pp);
        x[i] = -z; x[n - 1 - i] = z;
        w[i] = weight; w[n - 1 - i] = weight;
    }
    return (x, w);
}
var regle = Gauss(256);
double Integrer(Func<double, double> f, double a, double b)
{
    double milieu = (a + b) / 2, demi = (b - a) / 2, somme = 0;
    for (int i = 0; i < regle.x.Length; i++)
        somme += regle.w[i] * f(milieu + demi * regle.x[i]);
    return demi * somme;
}
double U(double x1, double x2, double x3) => x1 * x2 + x1 * x3 + x2 * x3;

double I_A(double s)
{
    // Sunrise d=2, voie A du carnet 02 (imbrication manuelle).
    // Les changements trigonométriques adoucissent les coins du simplexe.
    return Integrer(t => {
        double st = Math.Sin(t), ct = Math.Cos(t);
        double x1 = st * st, reste = ct * ct;
        double interieur = Integrer(v => {
            double sv = Math.Sin(v), cv = Math.Cos(v);
            double x2 = reste * sv * sv, x3 = reste * cv * cv;
            return 2 * reste * sv * cv / (U(x1, x2, x3) + s * x1 * x2 * x3);
        }, 0, Math.PI / 2);
        return 2 * st * ct * interieur;
    }, 0, Math.PI / 2);
}

// Référence mpmath du jumeau Python, et non résultat de la quadrature C#.
const double anchor = 2.343907238689458890602;
double val0 = I_A(0);
double ecartAncre = Math.Abs(val0 - anchor);
Console.WriteLine("I(0), voie A calculée en double = " + val0.ToString("G17", CultureInfo.InvariantCulture));
Console.WriteLine("Écart absolu à l'ancre du jumeau Python = " + ecartAncre.ToString("G17", CultureInfo.InvariantCulture));
Console.WriteLine("La double précision ne restitue pas les 22 chiffres affichés par mpmath.");
Debug.Assert(double.IsFinite(val0));

I(0), voie A calculée en double = 2.3439072445083626


Écart absolu à l'ancre du jumeau Python = 5.8189035812006296E-09


La double précision ne restitue pas les 22 chiffres affichés par mpmath.


## 2. La voie interne : l'intégrale en $x$ se fait à la main

Fixons $x_3 = \chi$, $x_1 = x(1-\chi)$, $x_2 = (1-x)(1-\chi)$ — jacobien
$|dx_1\,dx_2| = (1-\chi)\,dx\,d\chi$. Le dénominateur devient

$$U + s\,x_1x_2x_3 = (1-\chi)\big[\chi + b(\chi)\,x(1-x)\big],
\qquad b(\chi) = (1-\chi)(1+s\chi),$$

et le facteur $(1-\chi)$ du jacobien **s'annule contre celui du dénominateur** :

$$I(s) = \int_0^1 d\chi \int_0^1 \frac{dx}{\chi + b(\chi)\,x(1-x)}.$$

L'intégrale interne est élémentaire — avec $c = b + 4\chi$ :

$$\int_0^1 \frac{dx}{a + b\,x(1-x)} = \frac{4}{\sqrt{b\,c}}\;\operatorname{artanh}\!\sqrt{\frac{b}{c}}
\qquad (a = \chi,\ 0 < b < c \text{ sur } \chi \in (0,1),\ s>0),$$

d'où

$$I(s) = \int_0^1 \frac{4\,\operatorname{artanh}\sqrt{b/c}}{\sqrt{b\,c}}\;d\chi
\qquad\text{avec}\qquad y^2 = b(\chi)\,c(\chi).$$

Deux erreurs de facteur ont été attrapées par le planted truth ci-dessous avant
d'être commises : le coefficient vaut $4$ (pas $2$), et le $(1-\chi)$ du jacobien
disparaît — il ne reste **pas** dans l'intégrande. La forme finale parle d'elle-même :
une intégrale **de $\log$ contre $d\chi/y$**, la différentielle de première espèce
de la courbe $y^2 = bc$ — exactement la manière dont un dilogarithme classique est
$\int \log(1-x)\,dx/x$. Le rung elliptique n'est pas plaqué : il sort du calcul.

In [2]:
(double b, double c) b_c(double chi, double s)
{
    double b = (1 - chi) * (1 + s * chi);
    return (b, b + 4 * chi);
}
double FormuleInterne(double a, double b)
{
    double c = b + 4 * a;
    double racineB = Math.Sqrt(b), racineC = Math.Sqrt(c);
    // (sqrt(c)-sqrt(b)) est rationalisé par c-b = 4a.
    return 2 * Math.Log((racineC + racineB) * (racineC + racineB) / (4 * a)) / Math.Sqrt(b * c);
}
double I_C(double s)
{
    // Voie C : integrale interne analytique (artanh), reste en chi sur dchi/y.
    return Integrer(t => {
        double st = Math.Sin(t), ct = Math.Cos(t);
        double chi = st * st;
        var bc = b_c(chi, s);
        return 2 * st * ct * FormuleInterne(chi, bc.b);
    }, 0, Math.PI / 2);
}

// planted truth de la formule interne : quad direct vs formule fermee, a=0.3, b=1.4
double a0 = 0.3, b0 = 1.4, c0 = b0 + 4 * a0;
double lhs = Integrer(x => 1 / (a0 + b0 * x * (1 - x)), 0, 1);
double rhs = FormuleInterne(a0, b0);
double ecartFormule = Math.Abs(lhs - rhs) / rhs;
Console.WriteLine("formule interne : écart relatif calculé = " + ecartFormule.ToString("G17", CultureInfo.InvariantCulture));
Debug.Assert(ecartFormule < 1e-10);

// Références mpmath de la sortie du jumeau Python, distinctes des calculs.
var referencesAC = new Dictionary<int, double> {
    [1] = 2.236792700212646510529,
    [9] = 1.697946461195426121942,
    [25] = 1.225709127276724404979
};
var ecartsA = new Dictionary<int, double>();
var ecartsC = new Dictionary<int, double>();
Console.WriteLine("s | voie A calculée | voie C calculée | écart A-C | écart A-référence | écart C-référence");
foreach (int s in new[] { 1, 9, 25 })
{
    double a = I_A(s), c = I_C(s), reference = referencesAC[s];
    ecartsA[s] = Math.Abs(a - reference);
    ecartsC[s] = Math.Abs(c - reference);
    Console.WriteLine(string.Format(CultureInfo.InvariantCulture, "{0} | {1:G17} | {2:G17} | {3:G17} | {4:G17} | {5:G17}", s, a, c, Math.Abs(a - c), ecartsA[s], ecartsC[s]));
    Debug.Assert(double.IsFinite(a) && double.IsFinite(c));
}
Console.WriteLine("Les valeurs calculées en double ne reproduisent pas les 22 chiffres mpmath ; les écarts ci-dessus sont mesurés.");

formule interne : écart relatif calculé = 9.0432176275532622E-16


s | voie A calculée | voie C calculée | écart A-C | écart A-référence | écart C-référence


1 | 2.2367927060315509 | 2.2367927013578752 | 4.6736756686982517E-09 | 5.8189044693790493E-09 | 1.1452288006807976E-09


9 | 1.6979464670143298 | 1.6979464623406544 | 4.6736754466536468E-09 | 5.8189038032452345E-09 | 1.1452283565915877E-09


25 | 1.2257091330956285 | 1.2257091284219535 | 4.6736750025644369E-09 | 5.8189040252898394E-09 | 1.1452290227254025E-09


Les valeurs calculées en double ne reproduisent pas les 22 chiffres mpmath ; les écarts ci-dessus sont mesurés.


## 3. La courbe $y^2 = b(\chi)\,c(\chi)$ : racines et dégénérescences

$b$ a les racines $1$ et $-1/s$ ; $c$ (discriminant $(s+1)(s+9)$) les racines
$r_\pm = \frac{(3+s) \pm \sqrt{(s+1)(s+9)}}{2s}$. Quatre racines réelles
distinctes pour $s > 0$ : une quartique de genre 1 — **la courbe elliptique du
sunrise, en forme de Jacobi**. Deux dégénérescences la bornent :

- **$s \to 0$** : la racine $-1/s$ s'échappe à l'infini, la quartique perd un
  facteur — le genre tombe à 0. Le monde elliptique se réduit au monde du
  dilogarithme : l'ancre $I(0) = \frac{4}{\sqrt3}\mathrm{Cl}_2(\pi/3)$ vit
  **à la frontière**.
- **$s \to \infty$** : $r_- = -\frac1s + \frac{4}{s^2}$ et $-1/s$ — les deux
  racines négatives se collisionnent en $0$, l'écart entre elles tombant en
  $4/s^2$ : la pointe (cusp) de la famille.

In [3]:
double[] racines(double s)
{
    // Quatre racines de la quartique, triees croissantes.
    double d = Math.Sqrt((s + 1) * (s + 9));
    double rPlus = (s + 3 + d) / (2 * s);
    // Produit des deux racines de c : -1/s ; évite l'annulation dans rMoins.
    double rMoins = -1 / (s * rPlus);
    var r = new[] { -1 / s, 1.0, rMoins, rPlus };
    Array.Sort(r);
    return r;
}

// Valeurs affichées par le jumeau Python, arrondies dans sa sortie.
var referencesRacines = new double[][] {
    new[] { -100.0, -0.33186003, 1.0, 301.33186 },
    new[] { -0.25, -0.13278222, 1.0, 1.8827822 },
    new[] { -0.1111111111111111, -0.078689326, 1.0, 1.4120227 },
    new[] { -0.0625, -0.050485254, 1.0, 1.2379853 },
    new[] { -1e-6, -9.99996e-7, 1.0, 1.000004 }
};
var valeursS = new[] { 0.01, 4.0, 9.0, 16.0, 1e6 };
var ecartsRacines = new List<double>();
for (int i = 0; i < valeursS.Length; i++)
{
    double s = valeursS[i];
    double[] r = racines(s);
    double ecart = r.Zip(referencesRacines[i], (calcul, reference) => Math.Abs(calcul - reference)).Max();
    ecartsRacines.Add(ecart);
    Console.WriteLine("s=" + s.ToString("G17", CultureInfo.InvariantCulture) + " : racines calculées = [" + string.Join(", ", r.Select(x => x.ToString("G17", CultureInfo.InvariantCulture))) + "] ; écart maximal aux valeurs arrondies du jumeau = " + ecart.ToString("G17", CultureInfo.InvariantCulture));
    Debug.Assert(r[0] < r[1] && r[1] < r[2] && r[2] < r[3]);
}
Console.WriteLine("s->0   : -1/s fuit a -infini (racine qui s'echappe)");
Console.WriteLine("s->inf : -1/s et r_- se collisionnent en 0 (pointe)");

s=0.01 : racines calculées = [-100, -0.33186002963697453, 1, 301.33186002963697] ; écart maximal aux valeurs arrondies du jumeau = 2.9636964882229222E-08


s=4 : racines calculées = [-0.25, -0.13278221853731872, 1, 1.8827822185373186] ; écart maximal aux valeurs arrondies du jumeau = 1.8537318569045169E-08


s=9 : racines calculées = [-0.1111111111111111, -0.078689325833263227, 1, 1.4120226591665965] ; écart maximal aux valeurs arrondies du jumeau = 4.0833403547324565E-08


s=16 : racines calculées = [-0.0625, -0.050485254002759467, 1, 1.2379852540027594] ; écart maximal aux valeurs arrondies du jumeau = 4.5997240682993379E-08


s=1000000 : racines calculées = [-9.9999999999999995E-07, -9.9999600001999999E-07, 1, 1.000003999996] ; écart maximal aux valeurs arrondies du jumeau = 3.999911513119514E-12


s->0   : -1/s fuit a -infini (racine qui s'echappe)


s->inf : -1/s et r_- se collisionnent en 0 (pointe)


## 4. Les périodes : mesurées, puis réduites à $K(k)$

Avec $e_1 < e_2 < e_3 < e_4$ les racines triées, les deux cycles réels se
lisent sur les intervalles où $bc$ garde un signe : $bc > 0$ sur $(e_2, e_3)$
(c'est le domaine $\chi \in [0,1]$ de l'intégrale !) et $bc < 0$ sur
$(e_1, e_2)$. On mesure

$$\psi_1 = \int_{e_2}^{e_3} \frac{d\chi}{\sqrt{b\,c}}, \qquad
\psi_2 = \int_{e_1}^{e_2} \frac{d\chi}{\sqrt{-b\,c}},$$

mesurées sous la substitution $\chi = m + h\sin\theta$ (avec $m, h$ le milieu et
la demi-largeur de l'intervalle), qui donne exactement
$d\chi/\sqrt{(\chi-\mathrm{lo})(\mathrm{hi}-\chi)} = d\theta$ : la singularité
$1/\sqrt{\ }$ des deux bouts disparaît et l'intégrande devient lisse (facteur des
deux racines restantes, chacune rendue positive sur l'intervalle) — sans elle, la
quadrature directe ne rend pas le bon rapport du tout : l'écart relatif
dépasse l'unité, et non la précision demandée.

et la réduction de Legendre d'une quartique à quatre racines réelles relie les
deux au module $k^2 = \frac{(e_4-e_3)(e_2-e_1)}{(e_4-e_2)(e_3-e_1)}$ :

$$\frac{\psi_1}{\psi_2} = \frac{K(1-k^2)}{K(k^2)}.$$

La première écriture de cette section avait le rapport **inversé** — le planted
truth l'a attrapée ($1/0{,}4140 = 2{,}4156$ : c'est bien $\psi_1/\psi_2$ qui vaut
$K(1-k^2)/K(k^2)$), et c'est la version mesurée qui est commise ici. Le module
$k^2$ se mesure aussi **directement par birapport des racines** : deux voies
indépendantes pour le même nombre.

In [4]:
(double psi1, double psi2) periodes(double s)
{
    // psi1 (reel, e2..e3) et psi2 (perpendiculaire, e1..e2) par quadrature.
    // Substitution chi = m + h*sin(theta) : dchi/sqrt((chi-lo)(hi-chi)) = dtheta,
    // la singularite 1/sqrt aux deux bouts disparait, l'integrande devient lisse.
    // Les deux racines restantes encadrent l'intervalle : chaque facteur (chi - o)
    // est rendu positif par un signe fixe au milieu.
    var e = racines(s);
    double Psi(double lo, double hi, double autre1, double autre2)
    {
        double mid = (lo + hi) / 2, half = (hi - lo) / 2;
        return Integrer(theta => {
            double chi = mid + half * Math.Sin(theta);
            return 1 / Math.Sqrt(Math.Abs((chi - autre1) * (chi - autre2)));
        }, -Math.PI / 2, Math.PI / 2);
    }
    return (Psi(e[1], e[2], e[0], e[3]), Psi(e[0], e[1], e[2], e[3]));
}

double k2_module(double s)
{
    // Module de Legendre par birapport des racines.
    // Les deux petits écarts sont rationalisés pour éviter les annulations.
    double d = Math.Sqrt((s + 1) * (s + 9));
    double plusMoinsUn = 8 / (d + s - 3);
    double rPlus = 1 + plusMoinsUn;
    double ecartNegatif = plusMoinsUn / (s * rPlus);
    double e1 = -1 / s, e2 = -1 / (s * rPlus);
    return plusMoinsUn * ecartNegatif / ((rPlus - e2) * (1 - e1));
}

double Kcomplet(double m)
{
    // Moyenne arithmético-géométrique : K(m) = pi/(2 AGM(1,sqrt(1-m))).
    double a = 1, b = Math.Sqrt(1 - m);
    for (int i = 0; i < 50; i++)
    {
        double suivantA = (a + b) / 2, suivantB = Math.Sqrt(a * b);
        if (Math.Abs(suivantA - suivantB) <= 1e-15 * suivantA)
            return Math.PI / (2 * suivantA);
        a = suivantA;
        b = suivantB;
    }
    return Math.PI / (2 * a);
}

// Références mpmath du jumeau : valeurs affichées, pas résultats C#.
var referencesPeriodes = new Dictionary<int, (double k2, double ratio, double legendre)> {
    [9] = (0.00806504495005, 2.4155769875494845698, 2.4155769875494845858),
    [16] = (0.00208862860188, 2.8465788237216816068, 2.8465788237216816068),
    [25] = (0.00066981061043, 3.2088085228959175267, 3.2088085228959175373)
};
var ecartsPeriodes = new Dictionary<int, double>();
foreach (int s in new[] { 9, 16, 25 })
{
    var p = periodes(s);
    double k2 = k2_module(s), mesure = p.psi1 / p.psi2;
    double legendre = Kcomplet(1 - k2) / Kcomplet(k2);
    var reference = referencesPeriodes[s];
    ecartsPeriodes[s] = Math.Abs(mesure - reference.ratio);
    Console.WriteLine(string.Format(CultureInfo.InvariantCulture,
        "s={0} : k² calculé={1:G17}, écart à la référence arrondie={2:G17}",
        s, k2, Math.Abs(k2 - reference.k2)));
    Console.WriteLine(string.Format(CultureInfo.InvariantCulture,
        "  rapport mesuré={0:G17}, écart à la référence={1:G17}", mesure, ecartsPeriodes[s]));
    Console.WriteLine(string.Format(CultureInfo.InvariantCulture,
        "  rapport de Legendre={0:G17}, écart à la référence={1:G17}, écart relatif entre voies={2:G17}",
        legendre, Math.Abs(legendre - reference.legendre), Math.Abs(mesure - legendre) / legendre));
    Debug.Assert(double.IsFinite(mesure) && double.IsFinite(legendre));
}
Console.WriteLine("La précision double ne restitue pas tous les chiffres mpmath ; les écarts sont mesurés ci-dessus.");

s=9 : k² calculé=0.008065044950046266, écart à la référence arrondie=3.7348596437780657E-15


  rapport mesuré=2.4155769875494837, écart à la référence=8.8817841970012523E-16


  rapport de Legendre=2.4155769875494864, écart à la référence=1.7763568394002505E-15, écart relatif entre voies=1.1030636874063982E-15


s=16 : k² calculé=0.0020886286018811317, écart à la référence arrondie=1.1314733872058724E-15


  rapport mesuré=2.8465788237216771, écart à la référence=4.4408920985006262E-15


  rapport de Legendre=2.8465788237216891, écart à la référence=7.5495165674510645E-15, écart relatif entre voies=4.2122173347285452E-15


s=25 : k² calculé=0.0006698106104295916, écart à la référence arrondie=4.0841895837528952E-16


  rapport mesuré=3.2088085228959105, écart à la référence=7.1054273576010019E-15


  rapport de Legendre=3.2088085228959189, écart à la référence=1.3322676295501878E-15, écart relatif entre voies=2.6295414409882742E-15


La précision double ne restitue pas tous les chiffres mpmath ; les écarts sont mesurés ci-dessus.


## 5. $\tau(s)$ court : la courbe dépend de l'échelle

Le rapport des périodes définit le paramètre modulaire
$\tau(s) = i\,\psi_1/\psi_2$. Dans les carnets 01 et 02, tout se réduisait à
des constantes fixées ($\mathrm{Cl}_2$ de quelques angles) ; ici **la courbe
elle-même dépend de $s$** — c'est la leçon du rung elliptique : les périodes
remplacent les constantes, et $\tau(s)$ n'est constant nulle part. Au carnet
02, l'hypothèse naïve $I(s) = A\,K(k)$ à module
$k^2 = (1-\sqrt{1-9/s})/2$ (réel pour $s \ge 9$) échouait déjà ($A$ refusait d'être constant) — la
mesure ci-dessous montre que **ce module naïf n'est pas non plus celui de la
quartique** : deux familles différentes, la mesure tranche.

In [5]:
// Références mpmath du tableau du jumeau Python.
var referencesModules = new Dictionary<double, (double quartique, double naive, double tau)> {
    [9] = (0.008065044950046265, 0.5, 2.415576987549484569804),
    [16] = (0.002088628601881131, 0.1692810861169261761873, 2.846578823721681606773),
    [25] = (0.0006698106104295918, 0.1, 3.20880852289591752668),
    [100] = (1.424739782031008e-5, 0.02303039929152717542369, 4.434539869325926435733),
    [1e4] = (1.598081822369432e-11, 0.0002250506477940725316837, 8.795609024796386378584)
};
var ecartsTau = new Dictionary<double, double>();
Console.WriteLine("s | k² quartique calculé | k² naïf calculé | psi1/psi2 calculé | écarts absolus aux références Python");
foreach (double s in new[] { 9.0, 16.0, 25.0, 100.0, 1e4 })
{
    double k2q = k2_module(s);
    // Forme rationalisée du module naïf pour éviter une soustraction de nombres proches.
    double k2n = (9 / s) / (2 * (1 + Math.Sqrt(1 - 9 / s)));
    var p = periodes(s);
    double tau = p.psi1 / p.psi2;
    var reference = referencesModules[s];
    ecartsTau[s] = Math.Abs(tau - reference.tau);
    Console.WriteLine(string.Format(CultureInfo.InvariantCulture,
        "{0:G} | {1:G17} | {2:G17} | {3:G17} | {4:G17} ; {5:G17} ; {6:G17}",
        s, k2q, k2n, tau, Math.Abs(k2q - reference.quartique),
        Math.Abs(k2n - reference.naive), ecartsTau[s]));
    Debug.Assert(double.IsFinite(k2q) && double.IsFinite(k2n) && double.IsFinite(tau));
}
Console.WriteLine("La précision double ne restitue pas tous les chiffres mpmath ; les écarts sont mesurés ci-dessus.");

// la pointe : k^2 ~ 16/s^3 (ecart 4/s^2 entre les racines qui collisionnent, fois e4-e3 ~ 4/s)
// controle derive : la cible 1 et la tolerance sont posees ici, chaque ligne
// compare la valeur fraiche a la cible et rend son verdict (aucune affirmation
// sans nombre derive).
double cible = 1, tol = 2e-3;
var referencesPointe = new Dictionary<double, double> {
    [1e4] = 0.998801138981,
    [1e6] = 0.999988000114,
    [1e8] = 0.99999988
};
Console.WriteLine("Contrôle de pointe : cible = 1 ; tolérance = 0.002");
foreach (double s in new[] { 1e4, 1e6, 1e8 })
{
    double valeur = k2_module(s) * s * s * s / 16;
    double ecart = Math.Abs(valeur - cible);
    bool verdict = ecart <= tol;
    Console.WriteLine(string.Format(CultureInfo.InvariantCulture,
        "s={0:G} : valeur={1:G17} | écart à la cible={2:G17} | écart à la référence arrondie={3:G17} | conforme={4}",
        s, valeur, ecart, Math.Abs(valeur - referencesPointe[s]), verdict));
    Debug.Assert(verdict);
}

s | k² quartique calculé | k² naïf calculé | psi1/psi2 calculé | écarts absolus aux références Python


9 | 0.008065044950046266 | 0.5 | 2.4155769875494837 | 1.7347234759768071E-18 ; 0 ; 8.8817841970012523E-16


16 | 0.0020886286018811317 | 0.16928108611692616 | 2.8465788237216771 | 8.6736173798840355E-19 ; 2.7755575615628914E-17 ; 4.4408920985006262E-15


25 | 0.0006698106104295916 | 0.099999999999999992 | 3.2088085228959105 | 2.1684043449710089E-19 ; 1.3877787807814457E-17 ; 7.1054273576010019E-15


100 | 1.4247397820310071E-05 | 0.023030399291527174 | 4.4345398693259446 | 8.4703294725430034E-21 ; 0 ; 1.7763568394002505E-14


10000 | 1.5980818223694314E-11 | 0.00022505064779407253 | 8.7953325309028081 | 6.4623485355705287E-27 ; 0 ; 0.00027649389357797816


La précision double ne restitue pas tous les chiffres mpmath ; les écarts sont mesurés ci-dessus.


Contrôle de pointe : cible = 1 ; tolérance = 0.002


s=10000 : valeur=0.99880113898089462 | écart à la cible=0.0011988610191053795 | écart à la référence arrondie=1.0536016503692736E-13 | conforme=True


s=1000000 : valeur=0.99998800011399924 | écart à la cible=1.199988600075752E-05 | écart à la référence arrondie=7.7715611723760958E-16 | conforme=True


s=100000000 : valeur=0.99999988000001172 | écart à la cible=1.1999998827949554E-07 | écart à la référence arrondie=1.1768364061026659E-14 | conforme=True


## 6. Refus documentés (anneau déclaré avant recherche)

- **La forme fermée d'Adams–Bogner–Weinzierl** ($I$ en dilogarithmes
  elliptiques multiples) est **citée, pas reproduite** — la reproduire exigerait
  la construction du bidilogarithme elliptique, hors du périmètre d'un carnet
  de mesure. Notre ancre de fermeture reste $I(0)$.
- **Aucune constante fermée n'est identifiée pour $I(s)$, $s > 0$** : les
  sorties ci-dessus sont des mesures, pas des identifications. Chercher
  $I(9) = \alpha\,K(k) + \beta$ à $\alpha,\beta$ « simples » sans dérivation
  serait fabriquer un résultat.
- **Le module naïf du carnet 02 n'est pas celui de la quartique** (mesure §5) :
  c'est une question de corps de modules (courbes isogènes = modules reliés par
  transformations de Landen), pas une erreur à corriger dans l'un ou l'autre
  sens.

In [6]:
// Verification finale : la voie C reproduit l'ancre a s=0 par limite
// Références mpmath du jumeau Python, distinctes des valeurs calculées ici.
var referencesLimite = new Dictionary<double, double> {
    [0.1] = 2.3325243164695720804,
    [0.01] = 2.3427616936616913665,
    [0.001] = 2.3437926110735361935
};
var ecartsLimite = new Dictionary<double, double>();
foreach (double s in new[] { 0.1, 0.01, 0.001 })
{
    double c = I_C(s);
    double ecartAncreRelatif = Math.Abs(c - anchor) / anchor;
    ecartsLimite[s] = Math.Abs(c - referencesLimite[s]);
    Console.WriteLine(string.Format(CultureInfo.InvariantCulture,
        "I_C({0:G}) calculée = {1:G17} | écart relatif à l'ancre = {2:G17} | écart absolu à la référence Python = {3:G17}",
        s, c, ecartAncreRelatif, ecartsLimite[s]));
    Debug.Assert(double.IsFinite(c));
}
Console.WriteLine("La précision double ne restitue pas tous les chiffres mpmath ; les écarts sont mesurés ci-dessus.");
Console.WriteLine("(l'ecart decroit comme O(s) : la courbe degenere vers le monde Cl2 a la limite s=0)");

I_C(0.1) calculée = 2.3325243176148001 | écart relatif à l'ancre = 0.0048563871840864685 | écart absolu à la référence Python = 1.1452279125023779E-09


I_C(0.01) calculée = 2.3427616948069208 | écart relatif à l'ancre = 0.00048873260154219905 | écart absolu à la référence Python = 1.1452292447700074E-09


I_C(0.001) calculée = 2.343792612218766 | écart relatif à l'ancre = 4.8904013265105734E-05 | écart absolu à la référence Python = 1.1452296888592173E-09


La précision double ne restitue pas tous les chiffres mpmath ; les écarts sont mesurés ci-dessus.


(l'ecart decroit comme O(s) : la courbe degenere vers le monde Cl2 a la limite s=0)


## 7. Exercices

Trois exercices, à compléter — le carnet s'exécute de bout en bout sans eux
(règle C.1).

In [7]:
// Exercice 1. Rederiver la formule interne sur un autre couple (a, b).
// Verifier la quadrature de 1/(a + b*x*(1-x)) sur [0,1] contre 4*atanh(sqrt(b/c))/sqrt(b*c).
// avec c = b + 4a, pour a = 0.17, b = 2.3, a 25 chiffres.
// Etape 1 : poser a_exo, b_exo, c_exo = b_exo + 4*a_exo et comparer les deux cotes.
// TODO etudiant
object resultat_exo1 = null;
Console.WriteLine("Exercice a completer : formule interne sur (0.17, 2.3)");

Exercice a completer : formule interne sur (0.17, 2.3)


In [8]:
// Exercice 2. Reproduire la reduction de Legendre a s = 36.
// Calculer psi1, psi2, k^2 par birapport, et verifier psi1/psi2 = K(1-k^2)/K(k^2).
// Etape 1 : appeler periodes(36) et k2_module(36), comparer au rapport des ellipk.
// TODO etudiant
object resultat_exo2 = null;
Console.WriteLine("Exercice a completer : Legendre a s=36");

Exercice a completer : Legendre a s=36


In [9]:
// Exercice 3. La collision a l'infini : ou sont les racines a s = 1e8 ?
// Afficher racines(1e8) et verifier que les deux racines negatives tendent
// toutes deux vers -1/s, leur ecart e2 - e1 tombant en 4/s^2.
// Etape 1 : afficher racines(1e8), puis comparer e2 - e1 a 4e-16.
// TODO etudiant
object resultat_exo3 = null;
Console.WriteLine("Exercice a completer : collision des racines a s=1e8");

Exercice a completer : collision des racines a s=1e8


## 8. Ce que ce carnet établit

- **La courbe elliptique du sunrise exhibée par le calcul** : l'intégrale
  interne en $x$ est élémentaire (`artanh`), et le reste vit sur
  $y^2 = b\,c$ — quartique de genre 1, racines $\{1, -1/s, r_\pm\}$
  explicites, forme de Jacobi. Aucune courbe n'a été recopiée d'une référence.
- **$I(s)$ comme dilogarithme elliptique en formation** :
  $I(s) = \int_0^1 4\operatorname{artanh}\sqrt{b/c}\; d\chi/y$ —
  une intégrale de $\log$ contre la différentielle de première espèce, la
  construction même des polylogarithmes elliptiques.
- **Les périodes mesurées et réduites** : $\psi_1/\psi_2 = K(1-k^2)/K(k^2)$
  vérifié sur trois points, module $k^2$ mesuré par birapport des racines —
  deux voies indépendantes d'accord.
- **$\tau(s)$ non constant** : la courbe dépend de l'échelle. Les
  dégénérescences bornent la famille : $s \to 0$ (racine qui s'échappe —
  retour au monde $\mathrm{Cl}_2$, ancre $I(0)$), $s \to \infty$ (collision
  de racines — pointe, $k^2 \sim 16/s^3$).

L'étage suivant du classement BootLoops (K3, Calabi–Yau) demanderait des
graphes plus lourds (sunset à trois boucles) — hors périmètre de la série
tel qu'arbitré sur #20428.

## À retenir

| Notion | Ce qu'elle dit | Où la voir |
|---|---|---|
| Courbe elliptique exhibée | l'intégrale interne en $x$ est élémentaire (`artanh`), le reste vit sur $y^2 = b\,c$ — quartique de genre 1, racines $\{1, -1/s, r_\pm\}$, forme de Jacobi | §3 |
| Deux voies d'accord | $\psi_1/\psi_2 = K(1-k^2)/K(k^2)$ vérifié sur trois points : l'écart relatif entre les voies est au plus $4{,}3 \times 10^{-15}$ en double précision ; $k^2$ est mesuré par birapport des racines | §4 |
| $\tau(s)$ non constant | la courbe dépend de l'échelle : la partie imaginaire de $\tau = i\,\psi_1/\psi_2$ passe de $2{,}4156$ ($s=9$) à $8{,}7953$ ($s=10^4$) dans le calcul C# ; à $s=10^4$, l'écart absolu à la référence Python est d'environ $2{,}76 \times 10^{-4}$ | §5 |
| Dégénérescences | $s \to 0$ : $-1/s$ s'échappe, le genre tombe à 0, retour au monde $\mathrm{Cl}_2$ ; $s \to \infty$ : collision de racines (pointe), $k^2 \sim 16/s^3$ — mesuré : $k^2 s^3/16 = 0{,}99999988$ à $s = 10^8$ | §3, §5 |
| Refus documentés | la forme fermée d'Adams–Bogner–Weinzierl est citée, pas reproduite ; **aucune constante fermée n'est identifiée pour $I(s)$, $s > 0$** — les sorties sont des mesures | §6 |

Les deux voies A et C (intégration directe double et intégrale interne analytique) présentent des écarts absolus d'environ $4{,}67 \times 10^{-9}$ pour $s = 1, 9, 25$. La vérification de la formule interne donne un écart relatif d'environ $9{,}04 \times 10^{-16}$ pour $a = 0{,}3$ et $b = 1{,}4$.

## Vérifiez votre compréhension

1. La mesure de $\psi_1/\psi_2$ exige la substitution $\chi = m + h\sin\theta$. Que fait-elle exactement, et que rend la quadrature directe sans elle ?

<details><summary>Réponse</summary>

Elle donne exactement $d\chi/\sqrt{(\chi-\mathrm{lo})(\mathrm{hi}-\chi)} = d\theta$ : la singularité $1/\sqrt{\ }$ des deux bouts de l'intervalle disparaît et l'intégrande devient lisse (le facteur des deux racines restantes garde un signe fixe au milieu). Sans elle, la quadrature directe **ne rend pas le bon rapport du tout** — l'écart relatif dépasse l'unité (§4), très loin de la précision demandée. Même geste qu'au carnet 03 : la faute géométrique se traite géométriquement, pas en augmentant la précision.

</details>

2. À $s = 9$, le module naïf du carnet 02 vaut $0{,}5$ et celui de la quartique $0{,}00806504495005$. Lequel des deux est faux ?

<details><summary>Réponse</summary>

**Aucun** : ce sont deux familles différentes. Le module naïf $k^2 = (1-\sqrt{1-9/s})/2$ servait d'hypothèse au carnet 02 (où $A$ refusait déjà d'être constant) ; celui de la quartique se mesure par birapport des racines, voie indépendante confirmée par la réduction de Legendre. La question est de **corps de modules** — des courbes isogènes ont des modules reliés par transformations de Landen — pas une erreur à corriger dans un sens ou l'autre (§5, §6 : refus documentés).

</details>

3. Comment la limite $s \to 0$ reconnecte-t-elle ce carnet au monde du dilogarithme ?

<details><summary>Réponse</summary>

La racine $-1/s$ s'échappe à l'infini, la quartique perd un facteur, le genre tombe à 0 — et la mesure suit : les valeurs de $I_C(s)$ calculées pour les trois paramètres du §6 convergent vers l'ancre $I(0) = \frac{4}{\sqrt3}\mathrm{Cl}_2(\pi/3)$, avec un écart en $O(s)$. L'ancre des carnets 01–02 vit **à la frontière** du monde elliptique.

</details>

## Pour aller plus loin

- **Carnets voisins** : [Integrales-02 — Sunrise, masses égales](Integrales-02-Sunrise-Masses-Egales-Python.ipynb) (l'hypothèse naïve dont le module est discuté ici) et [Integrales-03 — Bulle, première divergence](Integrales-03-Bulle-Premiere-Divergence-Python.ipynb) (l'étage polylogarithme juste en dessous dans le classement BootLoops).
- **Sous-série suivante** : [Jauge-01 — Théorème de Noether](../Jauge/Jauge-01-Theoreme-Noether-Python.ipynb) — l'arc Noether → Yang–Mills de la série commence : symétries continues et courants conservés, avant l'invariance de jauge U(1)/SU(2) et l'action de Wilson.
- **Jumeau Python** → [Integrales-04-Sunrise-Courbe-Elliptique-Python](Integrales-04-Sunrise-Courbe-Elliptique-Python.ipynb) — le carnet d'origine, en Python.
- **Fil de la série** : [#23161 — EPIC Série Physique numérique](https://github.com/jsboige/CoursIA/issues/23161) — toit des distillations « Physics with AI » ; l'étage K3/Calabi–Yau au-delà est hors périmètre tel qu'arbitré sur [#20428](https://github.com/jsboige/CoursIA/issues/20428).

## Attestation de parité

Les écarts mesurés par section figurent dans les sorties : ancre (§1), formule interne et voies A et C (§2), racines (§3), périodes et réduction de Legendre (§4), modules et pointe (§5), limite vers l'ancre (§6). Les références mpmath sont distinctes des résultats calculés en double ; leur comparaison ne constitue pas une égalité à tous les chiffres affichés par le jumeau Python.

> Carnet issu de l'EPIC #26534.